## 1. Import Required Libraries

In [2]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.embeddings import get_embedding_model
from src.evaluation import (
	evaluate_rag,
	load_similarity_model,
)
from src.rag_chain import (
	generate_answer_with_sources,
	load_llm,
)
from src.retrieval import create_retriever
from src.vector_store import load_vector_store


/Users/jeet/Documents/Development/Projects/AI-Engineering-Portfolio/pdf-rag-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Embedding Model

In [3]:
embedding_model = get_embedding_model()

print("Embedding model loaded successfully.")

Using device: mps


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9096.29it/s]


Embedding model loaded successfully.


## 3. Load FAISS Vector Store

In [4]:
vector_store = load_vector_store(
	embedding_model=embedding_model
)

print("FAISS vector store loaded successfully.")

FAISS vector store loaded successfully.


## 4. Create Retriever

In [5]:
retriever = create_retriever(
	vector_store=vector_store,
	k=4
)

print("Retriever created successfully.")

Retriever created successfully.


## 5. Load Gemini

In [6]:
llm = load_llm()

print("Gemini loaded successfully.")

Gemini loaded successfully.


## 6. Load Similarity Model

In [7]:
similarity_model = load_similarity_model()

print("Similarity model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9225.34it/s]


Similarity model loaded successfully.


## 7. Create Evaluation Dataset

In [8]:
evaluation_dataset = [
	{
		"question": "What architecture does BERT use?",
		"expected_answer": "BERT is based on the Transformer architecture."
	},
	{
		"question": "What is Masked Language Modeling?",
		"expected_answer": "Masked Language Modeling predicts masked words using surrounding context."
	},
	{
		"question": "What is Next Sentence Prediction?",
		"expected_answer": "Next Sentence Prediction determines whether one sentence follows another."
	},
]

## 8. Evaluate the RAG System

In [9]:
results = []

for sample in evaluation_dataset:
	rag_result = generate_answer_with_sources(
		question=sample["question"],
		retriever=retriever,
		llm=llm
	)

	evaluation = evaluate_rag(
		question=sample["question"],
		expected_answer=sample["expected_answer"],
		generated_answer=rag_result["answer"],
		similarity_model=similarity_model
	)

	results.append(evaluation)

## 9. Display Results

In [10]:
for result in results:
	print("=" * 80)
	print(f"Question : {result['question']}")
	print()
	print(f"Expected : {result['expected_answer']}")
	print()
	print(f"Generated : {result['generated_answer']}")
	print()
	print(f"Similarity : {result['similarity']}")
	print(f"Status : {result['status']}")

Question : What architecture does BERT use?

Expected : BERT is based on the Transformer architecture.

Generated : Based on the provided document, BERT's model architecture is a multi-layer bidirectional Transformer encoder.

Similarity : 0.7865
Status : Good
Question : What is Masked Language Modeling?

Expected : Masked Language Modeling predicts masked words using surrounding context.

Generated : Based on the provided document, Masked Language Modeling (MLM)—also referred to as a Cloze task—is a pre-training procedure used to train deep bidirectional representations. It involves randomly masking a percentage of the input tokens and then predicting the original vocabulary ID of those masked tokens, rather than reconstructing the entire input.

Similarity : 0.6497
Status : Fair
Question : What is Next Sentence Prediction?

Expected : Next Sentence Prediction determines whether one sentence follows another.

Generated : Based on the provided document, Next Sentence Prediction (NSP) i

## 10. Display Summary

In [11]:
print("=" * 80)

excellent = sum(
	1 for result in results
	if result["status"] == "Excellent"
)

good = sum(
	1 for result in results
	if result["status"] == "Good"
)

fair = sum(
	1 for result in results
	if result["status"] == "Fair"
)

poor = sum(
	1 for result in results
	if result["status"] == "Poor"
)

average_similarity = (
	sum(result["similarity"] for result in results) / len(results)
)

print("Evaluation Summary")
print(f"Total Questions : {len(results)}")
print(f"Excellent : {excellent}")
print(f"Good : {good}")
print(f"Fair : {fair}")
print(f"Poor : {poor}")
print(f"Average Similarity : {average_similarity:.4f}")

Evaluation Summary
Total Questions : 3
Excellent : 0
Good : 1
Fair : 2
Poor : 0
Average Similarity : 0.6809


## 11. Evaluate a Custom Question

In [20]:
question = "What architecture does BERT use?"

expected_answer = "BERT is based on the Transformer architecture."

rag_result = generate_answer_with_sources(
	question=question,
	retriever=retriever,
	llm=llm,
)

evaluation = evaluate_rag(
	question=question,
	expected_answer=expected_answer,
	generated_answer=rag_result["answer"],
	similarity_model=similarity_model,
)

In [21]:
for key, value in evaluation.items():
	print(f"{key.capitalize()} -> {value}")

Question -> What architecture does BERT use?
Expected_answer -> BERT is based on the Transformer architecture.
Generated_answer -> Based on the provided document, BERT's model architecture is a multi-layer bidirectional Transformer encoder.
Similarity -> 0.7865
Status -> Good
